In [25]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv("../data/cleaned_marketing_AB.csv")
df

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...
588096,1278437,ad,False,1,Tuesday,23
588097,1327975,ad,False,1,Tuesday,23
588098,1038442,ad,False,3,Tuesday,23
588099,1496395,ad,False,1,Tuesday,23


## The Business Question

The marketing department has been running an ad campaign instead of showing users a generic Public Service Announcement (PSA) in that same ad space. 

This analysis uses a randomized A/B test dataset (~588,000 users) to answer questions and provide insight to leadership :
1. Did the ad campaign outperform the PSA group?
2. Is the difference statistically significant?
3. Does the frequency of ad exposure impact conversions?

**Dataset:** [Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) (Kaggle)

### Methodology

1. **Baseline comparison** — Compare raw conversion rates between ad and PSA groups.
2. **Significance testing** — Run a z-test to check if the difference is statistically real.
3. **Effect size** — Determine a confidence interval to see whether the difference is large enough to matter.
4. **Ad frequency** — How do conversion rate change with the number of impressions per user.
5. **The impact quantified** — Translate the results into an estimated number of additional conversions.

### Ad & PSA Group Analysis

In [27]:
summary = df.groupby("test_group")["converted"].agg(["sum", "count", "mean"])
summary.columns = ["conversions", "total_users", "conversion_rate"]
summary

,conversions,total_users,conversion_rate
test_group,,,
ad,14423,564577,0.025547
psa,420,23524,0.017854


We get a greater conversion rate with the ad test group, but with the vast difference in the amount of users for each group, the difference in conversion rate alone is not concrete enough evidence to say the ads are more valuable. 

In [28]:
from statsmodels.stats.proportion import proportions_ztest

ad_conversions = summary.loc["ad", "conversions"]
ad_total = summary.loc["ad", "total_users"]
psa_conversions = summary.loc["psa", "conversions"]
psa_total = summary.loc["psa", "total_users"]

count = [ad_conversions, psa_conversions]
nobs = [ad_total, psa_total]

z_stat, p_value = proportions_ztest(count, nobs)
print(f"z-statistic: {z_stat: .4f}")
print(f"p-value: {p_value: .6f}")


z-statistic:  7.3701
p-value:  0.000000


From those results I see that the differences between the two campaigns is signifcant. But, I need to ensure with ~588,000 users that these results are large enough to matter, because with a sample this large just a slight difference can produce a significant p value. So with that, I need to deduct the confidence interval to check whether the effect is truly large enough to matter at scale.

In [29]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_upp = confint_proportions_2indep(
    count1 = ad_conversions, nobs1 = ad_total,
    count2 = psa_conversions, nobs2 = psa_total
)

print(f"95% CI for the difference in conversions rates: ({ci_low: .4f}, {ci_upp: .4f})")

95% CI for the difference in conversions rates: ( 0.0059,  0.0094)


From the results I see that the difference is meaningful. We now know and can visually see statistically that the ad campaigns reliably outperform PSA's. 

Next I want to see if the frequency of ads matters.

In [30]:
df.groupby("test_group")["total_ads"].describe()

,count,mean,std,min,25%,50%,75%,max
test_group,,,,,,,,
ad,564577.0,24.823365,43.750456,1.0,4.0,13.0,27.0,2065.0
psa,23524.0,24.761138,42.860720,1.0,4.0,12.0,26.0,907.0


In [31]:
bins = [0, 1, 5, 10, 25, 50, 100, float("inf")]
labels = ["1", "2-5", "6-10", "11-25", "26-50", "51-100", "100+"]

df["ad_frequency_bucket"] = pd.cut(df["total_ads"], bins = bins, labels = labels)

ad_group = df[df["test_group"] == "ad"]

freq_summary = ad_group.groupby("ad_frequency_bucket")["converted"].agg(["sum", "count", "mean"])
freq_summary.columns = ["conversions", "total_users", "conversion_rate"]
freq_summary

,conversions,total_users,conversion_rate
ad_frequency_bucket,,,
1,86,54298,0.001584
2-5,341,115664,0.002948
6-10,386,79537,0.004853
11-25,1659,163135,0.010169
26-50,3037,85740,0.035421
51-100,5135,44149,0.116311
100+,3779,22054,0.171352


In [32]:
freq_summary.to_csv("ad_freq_summary.csv")

From the results we see the conversion rates climb steadily with greater ad impressions. From less than 1% conversion rate at just 1 ad viewed, to over 17% once hitting 100+ ads viewed.

There is a valid explanation for this growth as people who were already more engaged and potentially further along in a buying process naturally browse more, and because of that were served more ads and resulted in greater impressions. We can't ideally establish causation, it's simply rational to think that more engaged users generated more ad impressions through increased browsing activity rather than the ads solely driving the conversion.

### The Impact

This dataset doesn't include any revenue or price columns, regardless, I can quantify an impact with conversions.

In [33]:
ad_rate = ad_conversions / ad_total
psa_rate = psa_conversions / psa_total

additional_conversions = ad_total * (ad_rate - psa_rate)
print(f"Estimated additional conversions attributed to ads: {additional_conversions:.0f}")

Estimated additional conversions attributed to ads: 4343


Using the rate differences generated from before (0.0059, 0.0094), I can use these to give a range of estimated conversions.

In [34]:
low_estimate = ad_total * ci_low 
high_estimate = ad_total * ci_upp

print(f"95% CI range for additional conversions: {low_estimate:.0f} to {high_estimate:.0f}")

95% CI range for additional conversions: 3316 to 5284


So we can see that the ad campaign likely drove between ~3,300 - 5,300 additional conversions.

## Conclusions & Recommendations

**1. The ad campaign has shown to outdo the PSA group** 

A 95% CI of 0.59 - 0.94 percentage points, statistically significant at p < 0.001. 
Recommendation: Continue running ads over PSAs.

**2. Estimated impact** 

The campaign drove ~3,300 - 5,300 additional conversions compared to the PSA baseline.

**3. Ad frequency and conversion are strongly correlated, but likely not purely causal** 

Conversion rate climbs from under 1% at 1 ad shown, to over 17% with 100+ ads shown. 
But, before recommending any ad frequency number to hit, it's best to know that this relationship should be tested further with an additional experiment. That experiment should be dedicated to the ad frequencies, as more engaged users could simply be garnering more impressions rather than impressions driving the engagement/conversions.

**4. Next step**

Follow up experiment specifically examining ad frequency that would help separate correlation from causation.